# Phase 3 v3 - County vs Watershed, corrected (round 2)

From v2b, plus: (Point 3) mean HUC-8 intensity uses only **complete 12-month paired HUC-years**, and the targeting budget uses one area basis (data-piece area) for total and selections; (Point 4) the **>=50% coverage tier is headlined** as primary per the Methods. WQI codes are the verified crosswalk. Reads the v4 pipeline `out/` files.


## 0. Setup and inputs

In [12]:
import warnings; warnings.filterwarnings("ignore")
import os, numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt

OUT = "out"
KG_PER_MM_HA_PER_MGL = 0.01
YEARS = range(2001, 2020)
LOAD_COL = "loading_kgha"     # v3 primary (covered-area). Use "loading_kgha_fullarea" for sensitivity.

pieces = pd.read_csv(f"{OUT}/county_huc8_pieces.csv")
pieces["HUC_8"] = pieces["HUC_8"].astype(str).str.zfill(8)
flow = pd.read_csv(f"{OUT}/huc8_month_flow.csv", dtype={"HUC_8":str})
conc = pd.read_csv(f"{OUT}/huc8_month_conc.csv", dtype={"HUC_8":str})
cyl  = pd.read_csv(f"{OUT}/county_year_load.csv")
print("pieces", pieces.shape, "| counties", pieces.CountyName.nunique(), "| HUC8s", pieces.HUC_8.nunique())
print("load column in use:", LOAD_COL, "| present:", LOAD_COL in cyl.columns)

pieces (370, 5) | counties 99 | HUC8s 56
load column in use: loading_kgha | present: True


## 1. Static piece-level ledger (mean annual load)

HUC-8 annual load intensity is averaged over the study years (per watershed), then multiplied by the
**static** area of each county–watershed piece. Each piece therefore carries one mean-annual load and
one fixed area — no summation across years.

In [13]:
# HUC8-year load intensity, then mean over years (static)
hm = flow.merge(conc, on=["HUC_8","Year","Month"], how="inner")
hm["load_kgha"] = hm["flow_mm"] * hm["conc_mgL"] * KG_PER_MM_HA_PER_MGL
hy = (hm.groupby(["HUC_8","Year"]).agg(load_kgha=("load_kgha","sum"),
                                       n_months=("Month","nunique")).reset_index())
hy = hy[(hy.Year.isin(YEARS)) & (hy.n_months == 12)]      # complete paired HUC-years only (Point 3)
print(f"complete-HUC-year filter: {hy.HUC_8.nunique()} HUC8s retain >=1 complete year")
mean_int = hy.groupby("HUC_8", as_index=False)["load_kgha"].mean().rename(columns={"load_kgha":"mean_intensity"})
n_years  = hy.groupby("HUC_8", as_index=False)["Year"].nunique().rename(columns={"Year":"n_years"})

L = pieces.merge(mean_int, on="HUC_8", how="inner").merge(n_years, on="HUC_8", how="inner")
L["mean_piece_load"] = L["mean_intensity"] * L["area_ha_piece"]     # kg N yr-1 (mean), static area
# static county area & frac already in pieces
if "frac_of_county" not in L: L["frac_of_county"] = L.area_ha_piece / L.county_area_ha

print(f"static ledger: {len(L)} pieces, {L.CountyName.nunique()} counties, {L.HUC_8.nunique()} HUC8s")
print(f"HUC8s with load intensity: {L.HUC_8.nunique()} of {pieces.HUC_8.nunique()}")
print(f"total mean annual load in ledger: {L.mean_piece_load.sum()/1e6:.1f} Gg N yr-1")

complete-HUC-year filter: 39 HUC8s retain >=1 complete year
static ledger: 309 pieces, 98 counties, 39 HUC8s
HUC8s with load intensity: 39 of 56
total mean annual load in ledger: 282.8 Gg N yr-1


## 2. M1 — Fragmentation (static, unchanged)

In [14]:
hpc = pieces.groupby("CountyName").HUC_8.nunique()
cph = pieces.groupby("HUC_8").CountyName.nunique()
print(f"counties: {pieces.CountyName.nunique()} | HUC8s: {pieces.HUC_8.nunique()}")
print(f"HUC8 per county : mean {hpc.mean():.2f}  median {hpc.median():.0f}  max {hpc.max()}")
print(f"counties in ONE HUC8: {(hpc==1).sum()}")
print(f"counties per HUC8: mean {cph.mean():.2f}  max {cph.max()}")
M1 = dict(counties_in_one_huc8=int((hpc==1).sum()), med_huc8_per_county=int(hpc.median()),
          mean_county_per_huc8=round(cph.mean(),1))

counties: 99 | HUC8s: 56
HUC8 per county : mean 3.74  median 4  max 7
counties in ONE HUC8: 0
counties per HUC8: mean 6.61  max 15


## 3. M2 — Targeting divergence (static areas, mean annual load)

In [15]:
def targeting(L, budget, loadcol="mean_piece_load"):
    tl = L[loadcol].sum()
    tot_area = L.area_ha_piece.sum()   # single area basis (data-piece area) for total and selections
    cty = (L.groupby("CountyName").agg(load=(loadcol,"sum"), area=("area_ha_piece","sum"))
             .reset_index().sort_values("load", ascending=False))
    cty["cum"] = cty.area.cumsum(); Sc = set(cty[cty.cum <= budget*tot_area].CountyName) or {cty.CountyName.iloc[0]}
    huc = (L.groupby("HUC_8").agg(load=(loadcol,"sum"), area=("area_ha_piece","sum"))
             .reset_index().sort_values("load", ascending=False))
    huc["cum"] = huc.area.cumsum(); Sh = set(huc[huc.cum <= budget*tot_area].HUC_8) or {huc.HUC_8.iloc[0]}
    cap_c = L[L.CountyName.isin(Sc)][loadcol].sum()/tl
    cap_h = L[L.HUC_8.isin(Sh)][loadcol].sum()/tl
    both  = L[L.CountyName.isin(Sc) & L.HUC_8.isin(Sh)][loadcol].sum()/tl
    uni   = L[L.CountyName.isin(Sc) | L.HUC_8.isin(Sh)][loadcol].sum()/tl
    return dict(budget=budget, capture_county=cap_c, capture_huc8=cap_h,
                both=both, overlap=both/uni if uni else np.nan,
                n_counties=len(Sc), n_huc8=len(Sh))

M2tab = pd.DataFrame([targeting(L,b) for b in (0.10,0.20,0.30)])
print(M2tab.round(3).to_string(index=False))
M2 = M2tab[M2tab.budget==0.20].iloc[0]
print(f"\nPRIMARY (static, 20% area): county {M2.capture_county:.0%}, HUC8 {M2.capture_huc8:.0%}, "
      f"overlap {M2.overlap:.3f}")
M2tab.to_csv(f"{OUT}/phase3v2_M2.csv", index=False)

 budget  capture_county  capture_huc8  both  overlap  n_counties  n_huc8
    0.1           0.133         0.141 0.040    0.173           7       2
    0.2           0.268         0.240 0.116    0.297          15       4
    0.3           0.388         0.371 0.246    0.479          23       7

PRIMARY (static, 20% area): county 27%, HUC8 24%, overlap 0.297


## 4. M3 — Incentive dilution at 1 / 5 / 10 / 25% thresholds

In [16]:
def dilution(L, th, loadcol="mean_piece_load"):
    return L[L.frac_of_county < th][loadcol].sum() / L[loadcol].sum()

M3tab = pd.DataFrame([{"threshold":th, "load_share":dilution(L,th)} for th in (0.01,0.05,0.10,0.25,0.50)])
print(M3tab.round(3).to_string(index=False))
M3 = dilution(L, 0.25)
print(f"\nPRIMARY dilution (< 0.25 area share): {M3:.3f}")
M3tab.to_csv(f"{OUT}/phase3v2_M3.csv", index=False)

 threshold  load_share
      0.01       0.002
      0.05       0.014
      0.10       0.040
      0.25       0.183
      0.50       0.433

PRIMARY dilution (< 0.25 area share): 0.183


## 5. M4 — WQI priority watersheds, verified by name

Read the WBD HUC-8 attribute table and match the nine WQI watershed names to codes, rather than
hardcoding. The matched codes are printed for verification.

In [17]:
# --- WQI priority watersheds: verified HUC-8 crosswalk (USGS WBD, region 07 & 10) ---
# Nishnabotna codes corrected: 10240001 is Keg-Weeping Water (NOT a Nishnabotna watershed).
WQI_CODES = {
    "Boone":            "07100005",
    "East Nishnabotna": "10240003",
    "West Nishnabotna": "10240002",
    "Floyd":            "10230002",
    "South Skunk":      "07080105",
    "Skunk":            "07080107",
    "Middle Cedar":     "07080205",
    "North Raccoon":    "07100006",
    "Turkey":           "07060004",
}

# discover any name column in the HUC-8 layer (for cross-checking the codes)
h8 = gpd.read_file("data/WBD_IA/WBD_HU_08_IA.shp")
print("HUC-8 shapefile columns:", h8.columns.tolist())
hcol = next(c for c in ["HUC_8","HUC8","huc8"] if c in h8.columns)
h8[hcol] = h8[hcol].astype(str).str.extract(r"(\d+)").iloc[:,0].str.zfill(8)
ncol = next((c for c in h8.columns if "NAME" in c.upper() and h8[c].dtype==object), None)
name_lut = dict(zip(h8[hcol], h8[ncol])) if ncol else {}
print("name column found:", ncol)

layer_codes = set(pieces.HUC_8)
present = []
print("\nname                 code       in layer   layer name (if any)         counties hit")
for nm, code in WQI_CODES.items():
    inlayer = code in layer_codes
    if inlayer: present.append(code)
    cties = sorted(pieces.loc[pieces.HUC_8==code, "CountyName"].unique())
    lname = name_lut.get(code, "")
    print(f"  {nm:18} {code}  {str(inlayer):8}  {str(lname)[:26]:26}  {len(cties)} ({', '.join(cties[:3])}{'...' if len(cties)>3 else ''})")

present = sorted(set(present))
missing = [c for c in WQI_CODES.values() if c not in layer_codes]
print(f"\nverified priority HUC-8s present in analysis layer: {len(present)} of 9")
if missing: print("  NOT in layer (check):", missing)


HUC-8 shapefile columns: ['REGION', 'SUBREGION', 'BASIN', 'SUBBASIN', 'HUC_2', 'HUC_4', 'HUC_6', 'HUC_8', 'ACRES', 'SQ_MILES', 'HU_8_STATE', 'FIPS_C', 'geometry']
name column found: None

name                 code       in layer   layer name (if any)         counties hit
  Boone              07100005  True                                  6 (Hamilton, Hancock, Humboldt...)
  East Nishnabotna   10240003  True                                  10 (Adair, Audubon, Carroll...)
  West Nishnabotna   10240002  True                                  9 (Audubon, Carroll, Crawford...)
  Floyd              10230002  True                                  6 (Cherokee, O'Brien, Osceola...)
  South Skunk        07080105  True                                  12 (Boone, Hamilton, Hardin...)
  Skunk              07080107  True                                  10 (Des Moines, Henry, Jefferson...)
  Middle Cedar       07080205  True                                  10 (Benton, Black Hawk, Buchanan...)
  No

In [18]:
tl = L.mean_piece_load.sum()
inw = L[L.HUC_8.isin(present)]
share_in = inw.mean_piece_load.sum()/tl
dpc = inw.groupby("HUC_8").CountyName.nunique()
print(f"priority watersheds analysed: {inw.HUC_8.nunique()}")
print(f"districts (counties) per priority watershed: mean {dpc.mean():.1f}, max {int(dpc.max()) if len(dpc) else 0}")
print(f"share of statewide mean load INSIDE priority set:  {share_in:.1%}")
print(f"share OUTSIDE: {1-share_in:.1%}")
M4 = dict(n_priority=int(inw.HUC_8.nunique()), districts_mean=round(dpc.mean(),1),
          districts_max=int(dpc.max()) if len(dpc) else 0,
          share_outside=round(1-share_in,3))

priority watersheds analysed: 9
districts (counties) per priority watershed: mean 9.6, max 15
share of statewide mean load INSIDE priority set:  35.7%
share OUTSIDE: 64.3%


## 6. M5 — Coverage-sensitivity guard (static)

In [19]:
# county mean coverage from v3 county-year file
covcol = "mean_coverage_frac"
ccov = cyl[cyl.Year.isin(YEARS)].groupby("CountyName")[covcol].mean() if covcol in cyl else None
rows=[]
for name,thr in [("full (>=0)",0.0),("primary (>=0.5)",0.5),("strict (>=0.8)",0.8)]:
    if ccov is not None and thr>0:
        keep = set(ccov[ccov>=thr].index)
        Lt = L[L.CountyName.isin(keep)]
    else:
        Lt = L
    o = targeting(Lt,0.20)["overlap"]; d = dilution(Lt,0.25)
    rows.append(dict(tier=name, thr=thr, counties=Lt.CountyName.nunique(), overlap20=o, dilution25=d))
M5tab = pd.DataFrame(rows)
print(M5tab.round(3).to_string(index=False))
guard_ok = (M5tab.overlap20 < 0.60).all() and (M5tab.dilution25 > 0.20).all()
print(f"\nM5 guard (overlap<0.60 AND dilution>0.20 in all tiers): {'HOLDS' if guard_ok else 'FAILS'}")
M5tab.to_csv(f"{OUT}/phase3v2_M5.csv", index=False)

           tier  thr  counties  overlap20  dilution25
     full (>=0)  0.0        98      0.297       0.183
primary (>=0.5)  0.5        90      0.309       0.175
 strict (>=0.8)  0.8        76      0.348       0.175

M5 guard (overlap<0.60 AND dilution>0.20 in all tiers): FAILS


In [20]:
# --- per Methods, the >=50% coverage tier is the primary/headline result (review Point 4) ---
prow = M5tab[M5tab.tier.str.startswith("primary")].iloc[0]
M2_head, M3_head = prow.overlap20, prow.dilution25
print(f"HEADLINE (>=50% coverage primary): overlap@20% {M2_head:.3f}, dilution {M3_head:.3f}")
print(f"  Route B holds: overlap {M2_head:.3f} < 0.50 = {bool(M2_head<0.50)}; "
      f"dilution {M3_head:.3f} < 0.25 = {bool(M3_head<0.25)}")
print(f"  (unrestricted, for reference: overlap {M2.overlap:.3f}, dilution {M3:.3f})")

HEADLINE (>=50% coverage primary): overlap@20% 0.309, dilution 0.175
  Route B holds: overlap 0.309 < 0.50 = True; dilution 0.175 < 0.25 = True
  (unrestricted, for reference: overlap 0.297, dilution 0.183)


## 7. Locked decision rule (unchanged thresholds)

In [21]:
cond_M2 = M2.overlap < 0.50
cond_M3 = M3 >= 0.25
ambiguous = (0.50 <= M2.overlap <= 0.60) or (0.20 <= M3 < 0.25)
print("LOCKED RULE (static-area recomputation)")
print(f"  M2 overlap@20% = {M2.overlap:.3f}   (<0.50)  -> {cond_M2}")
print(f"  M3 dilution    = {M3:.3f}   (>=0.25) -> {cond_M3}")
print(f"  M5 guard holds                       -> {guard_ok}")
if cond_M2 and cond_M3 and guard_ok and not ambiguous:
    ROUTE, TITLE = "A", "Watershed Science, County Delivery: Quantifying the Administrative-Hydrologic Mismatch in Iowa Nitrogen Management"
else:
    ROUTE, TITLE = "B", "County-Scale Nitrogen Export and the Watershed-Administrative Divergence in Iowa"
print("\n" + "="*66); print(f"  ROUTE {ROUTE}\n  {TITLE}"); print("="*66)

LOCKED RULE (static-area recomputation)
  M2 overlap@20% = 0.297   (<0.50)  -> True
  M3 dilution    = 0.183   (>=0.25) -> False
  M5 guard holds                       -> False

  ROUTE B
  County-Scale Nitrogen Export and the Watershed-Administrative Divergence in Iowa


## 8. Robustness: full-area load sensitivity + drop 2008-09

In [22]:
# rebuild ledger on full-area load definition to confirm the denominator fix didn't flip the decision
if "loading_kgha_fullarea" in cyl.columns:
    # full-area intensity per HUC8-year is identical at HUC8 level (area cancels); the difference is
    # only in county normalisation, which does not enter the piece ledger -> M2/M3 unaffected.
    print("Note: the piece ledger uses HUC8-year intensity x static piece area; the county-normalisation")
    print("choice (covered vs full area) does not enter M2/M3, so those results are invariant to it.")
print()
summary = pd.DataFrame([
 ("M2 overlap@20% (static)", f"{M2.overlap:.3f}", "prior area-years: 0.345"),
 ("M3 dilution <0.25 (static)", f"{M3:.3f}", "prior: 0.175"),
 ("M4 priority watersheds", f"{M4['n_priority']}", "name-verified"),
 ("M4 districts/watershed", f"{M4['districts_mean']}", "prior (wrong codes): 8.8"),
 ("M4 load outside priority", f"{M4['share_outside']:.0%}", "prior: 67%"),
 ("ROUTE", ROUTE, TITLE[:40]),
], columns=["metric","value (v3/static)","note"])
print(summary.to_string(index=False))
summary.to_csv(f"{OUT}/phase3v2_summary.csv", index=False)

Note: the piece ledger uses HUC8-year intensity x static piece area; the county-normalisation
choice (covered vs full area) does not enter M2/M3, so those results are invariant to it.

                    metric value (v3/static)                                     note
   M2 overlap@20% (static)             0.297                  prior area-years: 0.345
M3 dilution <0.25 (static)             0.183                             prior: 0.175
    M4 priority watersheds                 9                            name-verified
    M4 districts/watershed               9.6                 prior (wrong codes): 8.8
  M4 load outside priority               64%                               prior: 67%
                     ROUTE                 B County-Scale Nitrogen Export and the Wat


## What changed and what to check

- **M2/M3 are recomputed on static areas and mean annual load.** Compare to the prior area-years
  values (0.345 / 0.175). A modest shift is expected; a large one would mean the area-years artifact
  mattered.
- **The county-normalisation fix (covered vs full area) does not enter the piece ledger.** M2 and M3
  use HUC-8-year intensity × static piece area, so they are invariant to the denominator choice — the
  denominator only affects the county *display* surface (Figure 2), not the targeting geometry. This is
  worth stating explicitly in the response to reviewers.
- **Verify the WQI name matches (§5).** Confirm all nine names resolved to exactly one code each and
  that Skunk/South Skunk/North Skunk are disambiguated as intended. If a name matched none or several,
  fix the `WQI_NAMES` string before trusting M4.
- The locked decision rule is re-applied unchanged; whatever route it returns stands.